# 02 · Trajectory

Demonstration of the 3-DOF reentry trajectory simulator implemented in `reentrykit.trajectory`. This notebook runs a representative ballistic reentry from 80 km at orbital velocity and visualizes the key flight quantities: altitude, velocity, Mach number, flight-path angle, dynamic pressure, and ground track.

The simulator integrates the Vinh-Busemann-Culp 3-DOF equations of motion (eqs. 2-44 to 2-49) using `scipy.integrate.solve_ivp` (RK45). It supports rotating or non-rotating planet models, time-varying L/D and bank-angle schedules, and Mach-dependent drag coefficients. This notebook exercises the simplest configuration: ballistic flight on non-rotating Earth.

References:
- Allen, H.J. and Eggers, A.J. (1958). *A Study of the Motion and Aerodynamic Heating of Ballistic Missiles Entering the Earth's Atmosphere*. NACA Report 1381.
- Vinh, N.X., Busemann, A., and Culp, R.D. (1980). *Hypersonic and Planetary Entry Flight Mechanics*. University of Michigan Press.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from reentrykit.trajectory import Vehicle, InitialState, simulate

# Reference vehicle: a generic biconic reentry body at sounding-rocket scale
vehicle = Vehicle(
    mass=500.0,  # kg
    reference_area=0.8,  # m^2
    drag_coefficient=1.5,  # -
    lift_to_drag_ratio=0.0,  # ballistic
    nose_radius=0.1,  # m
)

# Nominal entry: 80 km, 7.5 km/s (orbital), shallow -5 degree flight-path angle
initial_state = InitialState(
    altitude=80_000.0,  # m
    velocity=7500.0,  # m/s
    flight_path_angle=np.deg2rad(-5.0),  # rad
)

result = simulate(vehicle, initial_state)

# Summary print-out
print(f"Termination:     {result.termination_reason}")
print(f"Flight time:     {result.time[-1]:.1f} s")
print(f"Range:           {result.downrange[-1] / 1000:.1f} km")
print(f"Final altitude:  {result.altitude[-1]:.1f} m")
print(f"Final velocity:  {result.velocity[-1]:.1f} m/s")
print()
print(f"Peak dynamic pressure: {result.dynamic_pressure.max() / 1000:.1f} kPa "
      f"at {result.altitude[result.dynamic_pressure.argmax()] / 1000:.1f} km")
print(f"Peak Mach number:      {result.mach.max():.1f}")

# Peak deceleration
dV_dt = np.gradient(result.velocity, result.time)
peak_decel = -dV_dt.min()
i_peak_decel = dV_dt.argmin()
print(f"Peak deceleration:     {peak_decel:.1f} m/s^2 "
      f"({peak_decel / 9.80665:.1f} g) at "
      f"{result.altitude[i_peak_decel] / 1000:.1f} km")


In [ ]:
# Convert units for display
altitude_km = result.altitude / 1000.0
downrange_km = result.downrange / 1000.0
gamma_deg = np.rad2deg(result.flight_path_angle)
q_kpa = result.dynamic_pressure / 1000.0

fig, axes = plt.subplots(3, 2, figsize=(12, 12))

# Row 1, Col 1: altitude vs time
ax = axes[0, 0]
ax.plot(result.time, altitude_km, color="tab:blue", linewidth=1.8)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Altitude [km]")
ax.set_title("Altitude vs. Time")
ax.grid(True, alpha=0.3)

# Row 1, Col 2: velocity vs time
ax = axes[0, 1]
ax.plot(result.time, result.velocity, color="tab:red", linewidth=1.8)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Velocity [m/s]")
ax.set_title("Velocity vs. Time")
ax.grid(True, alpha=0.3)

# Row 2, Col 1: Mach vs altitude (the classical reentry plot)
ax = axes[1, 0]
ax.plot(result.mach, altitude_km, color="tab:green", linewidth=1.8)
ax.set_xlabel("Mach number [-]")
ax.set_ylabel("Altitude [km]")
ax.set_title("Mach-Altitude Profile")
ax.grid(True, alpha=0.3)
ax.invert_xaxis()  # read from high Mach at top right to low Mach at ground

# Row 2, Col 2: flight-path angle vs altitude
ax = axes[1, 1]
ax.plot(gamma_deg, altitude_km, color="tab:purple", linewidth=1.8)
ax.set_xlabel("Flight-path angle [deg]")
ax.set_ylabel("Altitude [km]")
ax.set_title("Flight-Path Angle vs. Altitude")
ax.grid(True, alpha=0.3)

# Row 3, Col 1: dynamic pressure vs altitude
ax = axes[2, 0]
ax.plot(q_kpa, altitude_km, color="tab:orange", linewidth=1.8)
ax.set_xlabel("Dynamic pressure [kPa]")
ax.set_ylabel("Altitude [km]")
ax.set_title("Dynamic Pressure vs. Altitude")
ax.grid(True, alpha=0.3)
# Annotate peak q
i_peak_q = result.dynamic_pressure.argmax()
ax.scatter([q_kpa[i_peak_q]], [altitude_km[i_peak_q]], color="black", zorder=5, s=40)
ax.annotate(
    f"Peak q = {q_kpa[i_peak_q]:.1f} kPa\nat {altitude_km[i_peak_q]:.1f} km",
    xy=(q_kpa[i_peak_q], altitude_km[i_peak_q]),
    xytext=(15, -20), textcoords="offset points", fontsize=9,
    arrowprops=dict(arrowstyle="->", color="black", alpha=0.5),
)

# Row 3, Col 2: ground track (altitude vs downrange)
ax = axes[2, 1]
ax.plot(downrange_km, altitude_km, color="tab:brown", linewidth=1.8)
ax.set_xlabel("Downrange [km]")
ax.set_ylabel("Altitude [km]")
ax.set_title("Trajectory Profile (Altitude vs. Range)")
ax.grid(True, alpha=0.3)

fig.suptitle(
    "Ballistic Reentry Trajectory — 80 km, 7500 m/s, γ₀ = -5°",
    fontsize=14,
    y=1.00,
)
fig.tight_layout()
plt.show()

## Observations

The trajectory exhibits the classical features of a hypersonic ballistic reentry:

- **Velocity history** shows the characteristic S-curve: minimal deceleration for the first ~50 seconds while the vehicle is in thin upper atmosphere, violent deceleration through the 60–120 s window where dynamic pressure peaks, then near-constant terminal velocity in the dense lower atmosphere.

- **Peak dynamic pressure** of 63.5 kPa occurs at 35.6 km altitude — consistent with the classical Allen-Eggers result that peak deceleration (and peak heating) for a ballistic reentry occurs at the altitude where ρ(h) / (m / (Cd · S)) is maximized.

- **Flight-path angle** starts at a shallow −5° but steepens dramatically as the vehicle loses horizontal velocity faster than vertical velocity. By 20 km the flight path is nearly vertical.

- **Mach number** decays from 27 at entry to subsonic near 20 km, with most of the deceleration occurring between 40 and 25 km.

These four quantities — Mach, dynamic pressure, altitude, and flight-path angle — define the aerothermal, structural, and control environments that downstream design modules (TPS sizing, structural loads, GNC design) must handle.

## Parameter Study: Entry Flight-Path Angle

A parameter sweep over entry flight-path angles from −2° (very shallow) to −15° (steep). Steeper entries deposit the vehicle into denser atmosphere at higher velocity, producing higher peak deceleration and peak heating but shorter range and shorter total flight time.

This is the fundamental design tradeoff for ballistic reentry: shallow entries are gentler on the vehicle but less precisely controllable in terms of landing location; steep entries give better targeting but require more robust TPS and structure.

In [ ]:
entry_angles_deg = [-2.0, -3.0, -5.0, -7.5, -10.0, -15.0]

results = []
for gamma_deg in entry_angles_deg:
    state = InitialState(
        altitude=80_000.0,
        velocity=7500.0,
        flight_path_angle=np.deg2rad(gamma_deg),
    )
    results.append(simulate(vehicle, state))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: altitude vs time for each entry angle
ax = axes[0]
for gamma_deg, r in zip(entry_angles_deg, results):
    ax.plot(r.time, r.altitude / 1000.0, linewidth=1.8, label=f"γ₀ = {gamma_deg}°")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Altitude [km]")
ax.set_title("Altitude vs. Time")
ax.grid(True, alpha=0.3)
ax.legend()

# Right: dynamic pressure vs altitude for each
ax = axes[1]
for gamma_deg, r in zip(entry_angles_deg, results):
    ax.plot(r.dynamic_pressure / 1000.0, r.altitude / 1000.0, linewidth=1.8, label=f"γ₀ = {gamma_deg}°")
ax.set_xlabel("Dynamic pressure [kPa]")
ax.set_ylabel("Altitude [km]")
ax.set_title("Dynamic Pressure vs. Altitude")
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle("Effect of Entry Flight-Path Angle", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

# Summary table
print(f"{'γ₀ [deg]':>8}  {'Time [s]':>9}  {'Range [km]':>11}  {'Peak q [kPa]':>13}  {'Peak decel [g]':>15}")
print("-" * 70)
for gamma_deg, r in zip(entry_angles_deg, results):
    dV_dt = np.gradient(r.velocity, r.time)
    peak_g = -dV_dt.min() / 9.80665
    print(f"{gamma_deg:>8.1f}  {r.time[-1]:>9.1f}  {r.downrange[-1]/1000:>11.1f}  "
          f"{r.dynamic_pressure.max()/1000:>13.1f}  {peak_g:>15.1f}")

## Key Takeaways

The parameter sweep quantifies the fundamental ballistic-reentry design tradeoff:

- **Peak deceleration scales approximately as sin(γ₀)**, as predicted by Allen-Eggers (1958). Doubling the entry angle roughly doubles the peak g-loading.
- **Range decreases rapidly with entry angle**: a −2° entry covers 977 km while a −15° entry covers only 228 km.
- **Peak dynamic pressure scales with entry angle**: 40 kPa at −2° versus 175 kPa at −15°.

A typical crewed reentry (Soyuz, Dragon) targets around −1.5° to −2° for gentle deceleration (~4–6 g) but trades precision for comfort. Uncrewed sample-return capsules (Stardust, Genesis) tolerate higher g-loading (~30+ g) in exchange for smaller landing footprints.

## Next Steps

With the atmosphere and trajectory modules complete and validated, the natural next module is **aerothermal analysis**: computing stagnation-point heating, flat-plate convective heating, and pressure distributions at each trajectory point. This unlocks TPS sizing, the central design problem for any reentry vehicle.

The atmosphere + trajectory + aerothermal triad is the complete preliminary-design pipeline for hypersonic reentry.